In [1]:
import os
import torch
import torch.nn.functional as F
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
import torch.multiprocessing as mp
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torch.utils.data.distributed import DistributedSampler
import time
import sys

def setup(rank, world_size):
    """Configure le processus distribué"""
    os.environ["MASTER_ADDR"] = "127.0.0.1"
    os.environ["MASTER_PORT"] = "29500"

    try:
        dist.init_process_group(
            backend="gloo",
            rank=rank,
            world_size=world_size,
            init_method='env://'
        )
        print(f"[Rank {rank}] Successfully initialized process group")
    except Exception as e:
        print(f"[Rank {rank}] Failed to initialize: {e}")
        sys.exit(1)

def cleanup():
    """Nettoie le processus distribué"""
    if dist.is_initialized():
        dist.destroy_process_group()

def get_dataloader(batch_size=64, distributed=False, rank=0, world_size=1):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])

    dataset = datasets.MNIST("./data", train=True, download=True, transform=transform)

    if distributed:
        sampler = DistributedSampler(dataset, num_replicas=world_size, rank=rank, shuffle=True)
        return DataLoader(dataset, batch_size=batch_size, sampler=sampler, num_workers=0), sampler
    else:
        return DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=0), None


import torch.nn as nn

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3)
        self.conv2 = nn.Conv2d(32, 64, 3)
        self.fc1 = nn.Linear(64 * 12 * 12, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.max_pool2d(x, 2)
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x


def train_worker(rank, world_size, epochs, batch_size):
    """Fonction d'entraînement pour chaque processus distribué"""
    try:
        print(f"[Rank {rank}] Worker started")

        # Setup du groupe de processus
        setup(rank, world_size)
        device = torch.device("cpu")

        # Création du modèle DDP
        model = SimpleCNN().to(device)
        ddp_model = DDP(model, device_ids=None)

        optimizer = torch.optim.Adam(ddp_model.parameters(), lr=0.001)
        loss_fn = torch.nn.CrossEntropyLoss()

        # Chargement des données
        loader, sampler = get_dataloader(
            batch_size=batch_size,
            distributed=True,
            rank=rank,
            world_size=world_size
        )

        if rank == 0:
            print(f"[Rank 0] Starting training with {len(loader)} batches per epoch")

        start = time.time()

        for epoch in range(epochs):
            if sampler is not None:
                sampler.set_epoch(epoch)

            ddp_model.train()
            total_loss = 0
            num_batches = 0

            for data, target in loader:
                data, target = data.to(device), target.to(device)

                optimizer.zero_grad()
                output = ddp_model(data)
                loss = loss_fn(output, target)

                loss.backward()
                optimizer.step()

                total_loss += loss.item()
                num_batches += 1

            avg_loss = total_loss / num_batches

            if rank == 0:
                print(f"[Rank 0] Epoch {epoch+1}/{epochs} | Loss: {avg_loss:.4f}")

        if rank == 0:
            elapsed = round(time.time() - start, 2)
            print(f"[Rank 0] Training completed in {elapsed} sec")

        cleanup()
        print(f"[Rank {rank}] Worker finished successfully")

    except Exception as e:
        print(f"[Rank {rank}] Error: {e}")
        import traceback
        traceback.print_exc()
        cleanup()
        sys.exit(1)


def train_single(epochs=2, batch_size=256):
    """Entraînement single-process pour comparaison"""
    device = torch.device("cpu")
    print("Using CPU only")

    loader, _ = get_dataloader(batch_size=batch_size)

    model = SimpleCNN().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    start = time.time()

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for data, target in loader:
            data, target = data.to(device), target.to(device)

            optimizer.zero_grad()
            output = model(data)
            loss = F.cross_entropy(output, target)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1} | Loss: {total_loss/len(loader):.4f}")

    print("Time:", round(time.time() - start, 2), "sec")


if __name__ == "__main__":
    # Force spawn method for compatibility
    mp.set_start_method('spawn', force=True)

    # Option 1: Entraînement single-process
    print("\n=== Single Process Training ===")
    train_single(epochs=2, batch_size=256)

    # # Option 2: Entraînement distribué
    # print("\n=== Distributed Training ===")
    # world_size = 2

    # try:
    #     mp.spawn(
    #         train_worker,
    #         args=(world_size, 2, 64),
    #         nprocs=world_size,
    #         join=True
    #     )
    #     print("\n=== Distributed training completed successfully ===")
    # except Exception as e:
    #     print(f"\n=== Distributed training failed: {e} ===")
    #     import traceback
    #     traceback.print_exc()


=== Single Process Training ===
Using CPU only


100%|██████████| 9.91M/9.91M [00:00<00:00, 63.0MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.68MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 14.6MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 7.57MB/s]


Epoch 1 | Loss: 0.1809
Epoch 2 | Loss: 0.0460
Time: 345.38 sec
